In [ ]:
#3.3
from collections import defaultdict
from datetime import datetime, timedelta
import json
import uuid
import time

#Имитация Kafka
class MockKafkaTopic:
    def __init__(self, name):
        self.name = name
        self.messages = []

    def send(self, message):
        self.messages.append(message)

    def consume(self):
        if self.messages:
            return self.messages.pop(0)
        return None

#Имитация топиков Kafka
transactions_raw = MockKafkaTopic('transactions.raw')
transactions_enriched = MockKafkaTopic('transactions.enriched')
fraud_alerts = MockKafkaTopic('fraud.alerts')
notifications = MockKafkaTopic('notifications')

#3.3: АНТИФРОД-СЕРВИС (CONSUMER + STREAM PROCESSING)
class FraudDetectionService:
    def __init__(self):
        #Stateful хранение для правила 3
        self.user_transactions = defaultdict(list)

    def check_rules(self, tx):
        #Правило 1: Сумма > 150 000 руб.
        if tx['amount'] > 150000:
            return True, f"amount_exceeds_limit: {tx['amount']} > 150000"

        #Правило 2: Транзакция типа transfer и сумма > 50 000 руб.
        if tx['type'] == 'transfer' and tx['amount'] > 50000:
            return True, f"large_transfer: {tx['amount']} > 50000"

        #Правило 3: Более 5 транзакций за 30 секунд (stateful обработка)
        now = datetime.now()
        cutoff = now - timedelta(seconds=30)

        #Очищаем старые транзакции
        self.user_transactions[tx['user_id']] = [
            t for t in self.user_transactions[tx['user_id']]
            if t['timestamp'] > cutoff]

        #Добавляем текущую
        self.user_transactions[tx['user_id']].append({'timestamp': now})

        #Проверяем количество
        if len(self.user_transactions[tx['user_id']]) > 5:
            return True, f"too_many_transactions: {len(self.user_transactions[tx['user_id']])} in last 30 seconds"

        return False, ""

    def process(self, tx):
        is_fraud, reason = self.check_rules(tx)

        if is_fraud:
            #Отправка в fraud.alerts
            alert = {"transaction": tx, "fraud_reason": reason, "detected_at": datetime.now().isoformat()}
            fraud_alerts.send(alert)

            #Отправка в notifications
            notifications.send(alert)

            #Обогащение транзакции
            enriched = {**tx, "fraud_status": "suspicious", "fraud_reason": reason}
        else:
            #Обогащение транзакции
            enriched = {**tx, "fraud_status": "clean", "fraud_reason": None}

        #Отправка в transactions.enriched
        transactions_enriched.send(enriched)
        return enriched

#CONSUMER: чтение из transactions.raw
def fraud_consumer():
    service = FraudDetectionService()

    while True:
        tx = transactions_raw.consume()
        if not tx:
            break
        service.process(tx)

#Тест пункта 3.3
#Имитация producer для transactions.raw
for i in range(7):
    tx = {"transaction_id": str(uuid.uuid4()),
        "user_id": "user_1",
        "amount": 100,
        "currency": "RUB",
        "type": "payment",
        "timestamp": datetime.now().isoformat(),
        "merchant_id": "merch_01",
        "location": "Moscow"}
    transactions_raw.send(tx)
    time.sleep(0.5)

#Запуск consumer
fraud_consumer()

print(f"\nРезультат")
print(f"transactions.enriched: {len(transactions_enriched.messages)} сообщений")
print(f"fraud.alerts: {len(fraud_alerts.messages)} сообщений")
print(f"notifications: {len(notifications.messages)} сообщений")


=== РЕЗУЛЬТАТ ПУНКТА 3.3 ===
transactions.enriched: 7 сообщений
fraud.alerts: 2 сообщений
notifications: 2 сообщений


In [ ]:
#3.4
#Условия: читает из transactions.enriched (созданного в пункте 3.3)
#Распределяет по топикам: crm.updates, ledger.events, notifications

from datetime import datetime
import json
import uuid

#Имитация Kafka-топиков (из пункта 3.1)
class MockKafkaTopic:
    def __init__(self, name):
        self.name = name
        self.messages = []

    def send(self, message):
        self.messages.append(message)

    def consume(self):
        if self.messages:
            return self.messages.pop(0)
        return None

#Топики согласно пункту 3.1 и 3.4
transactions_enriched = MockKafkaTopic('transactions.enriched')
crm_updates = MockKafkaTopic('crm.updates')
ledger_events = MockKafkaTopic('ledger.events')
notifications = MockKafkaTopic('notifications')

#3.4: CONTENT-BASED ROUTER
class ContentBasedRouter:
    def route(self, tx):
        if tx['type'] == 'purchase':
            crm_updates.send(tx)
            return "purchase -> crm.updates"

        elif tx['type'] == 'transfer':
            ledger_events.send(tx)
            return "transfer -> ledger.events"

        elif tx['type'] == 'payment':
            ledger_events.send(tx)
            notifications.send(tx)
            return "payment -> ledger.events + notifications"

        return f"unknown type: {tx['type']}"

#CONSUMER: читает из transactions.enriched
def router_consumer():
    router = ContentBasedRouter()

    while True:
        tx = transactions_enriched.consume()
        if not tx:
            break
        result = router.route(tx)
        print(f"{result} | {tx['transaction_id']}")

#Подготовка тестовых данных (на основе пункта 3.3)
test_enriched_transactions = [
    {"transaction_id": str(uuid.uuid4()),
        "user_id": "user_1",
        "amount": 1000,
        "currency": "RUB",
        "type": "purchase",
        "timestamp": datetime.now().isoformat(),
        "merchant_id": "merch_01",
        "location": "Moscow",
        "fraud_status": "clean",
        "fraud_reason": None},
    {"transaction_id": str(uuid.uuid4()),
        "user_id": "user_2",
        "amount": 5000,
        "currency": "RUB",
        "type": "transfer",
        "timestamp": datetime.now().isoformat(),
        "merchant_id": None,
        "location": "SPb",
        "fraud_status": "clean",
        "fraud_reason": None},
    {"transaction_id": str(uuid.uuid4()),
        "user_id": "user_3",
        "amount": 2000,
        "currency": "RUB",
        "type": "payment",
        "timestamp": datetime.now().isoformat(),
        "merchant_id": "merch_02",
        "location": "Kazan",
        "fraud_status": "clean",
        "fraud_reason": None}]

#Заполнение топика transactions.enriched
for tx in test_enriched_transactions:
    transactions_enriched.send(tx)

#Запуск
router_consumer()

print(f"\nРезультаты")
print(f"crm.updates: {len(crm_updates.messages)} сообщений (только purchase)")
print(f"ledger.events: {len(ledger_events.messages)} сообщений (transfer + payment)")
print(f"notifications: {len(notifications.messages)} сообщений (только payment)")

print("\nДетали")
print(f"crm.updates типы: {[m['type'] for m in crm_updates.messages]}")
print(f"ledger.events типы: {[m['type'] for m in ledger_events.messages]}")
print(f"notifications типы: {[m['type'] for m in notifications.messages]}")

=== ПУНКТ 3.4 - CONTENT-BASED ROUTER ===

purchase -> crm.updates | fa37d933-7691-4644-aaf7-9a62a3eba173
transfer -> ledger.events | fbf78d74-5591-4f0e-94ab-008d1c6ed2c5
payment -> ledger.events + notifications | 730aafad-cff3-4ebb-8872-f8283f10fb0a

=== РЕЗУЛЬТАТ ПУНКТА 3.4 ===
crm.updates: 1 сообщений (только purchase)
ledger.events: 2 сообщений (transfer + payment)
notifications: 1 сообщений (только payment)

=== ДЕТАЛИ ===
crm.updates типы: ['purchase']
ledger.events типы: ['transfer', 'payment']
notifications типы: ['payment']


## Обоснование реализации пунктов 3.3 и 3.4

В рамках выполнения практического задания по интеграции систем с использованием Apache Kafka были реализованы антифрод-сервис (пункт 3.3) и маршрутизатор на основе содержимого сообщений (пункт 3.4). Исходный план предполагал развертывание полноценного окружения с Zookeeper и брокером Kafka через Docker Compose, создание топиков и запуск генератора транзакций, однако на практике это столкнулось с техническими ограничениями. На локальной машине отсутствовал Docker, установка Homebrew на macOS 13 оказалась проблематичной из-за требований к Xcode, а попытки запуска Faust в Google Colab привели к ошибкам несовместимости версий библиотек. В связи с этим было принято решение реализовать имитационный подход, который сохраняет всю бизнес-логику задания и демонстрирует корректную работу всех требуемых механизмов.

Для пункта 3.3 разработан антифрод-сервис, который читает транзакции из топика transactions.raw и применяет три правила выявления подозрительных операций. Первое правило проверяет превышение суммы 150 000 рублей, второе выявляет переводы на сумму более 50 000 рублей, третье реализует stateful обработку для обнаружения более пяти транзакций от одного пользователя за последние 30 секунд. Stateful логика реализована вручную с использованием словаря в памяти, где для каждого пользователя хранится список временных меток его транзакций с периодической очисткой устаревших записей. Это полностью соответствует требованию задания, которое допускает как использование Faust, так и ручную реализацию. По результатам проверки безопасные транзакции обогащаются полем fraud_status: "clean" и отправляются в топик transactions.enriched, а подозрительные — в fraud.alerts с указанием причины и дублируются в notifications для оповещения службы безопасности.

Для пункта 3.4 реализован отдельный consumer, который читает обогащённые транзакции из топика transactions.enriched и распределяет их в зависимости от типа. Транзакции типа purchase направляются в crm.updates для начисления бонусов, transfer — в ledger.events для учёта, а payment реализует паттерн publish-subscribe с одновременной отправкой и в ledger.events, и в notifications. Все три сценария успешно отработали на тестовых данных.

Имитационная реализация полностью сохраняет логику взаимодействия с Kafka: созданы классы-заглушки для топиков с методами send и consume, producer генерирует транзакции в соответствии со спецификацией из пункта 3.2 с полями transaction_id, user_id, amount, currency, type, timestamp, merchant_id, location, а consumer обрабатывает их в цикле. Такой подход позволяет продемонстрировать работоспособность всех требуемых механизмов без развёртывания реальной инфраструктуры, что в данных условиях является оптимальным решением.